# Atera kidney: review candidate cells with Yearbook

Use a tissue-wide **Landscape** for context, then send candidate cell IDs to a
paginated **Yearbook**. Select the portraits that deserve a closer look and get
their IDs back in Python.

**Why EPCAM and PTPRC?** EPCAM marks epithelial cells, including kidney tubule
epithelium, and PTPRC encodes CD45, which marks immune cells. They belong to
separate lineages, so a single cell is not expected to express both. A cell that
does is worth a closer look: an immune cell pressed against a tubule may have been
segmented together with its epithelial neighbor (a segmentation merge), or a few
transcripts from a neighboring cell may have been assigned to the wrong cell. These
are **review candidates, not doublet calls**: coexpression alone cannot distinguish
technical artifacts from biology.

The data are the public [10x Genomics Atera WTA FFPE human non-diseased kidney
dataset](https://www.10xgenomics.com/datasets/atera-wta-ffpe-human-nd-kidney).
We use the same [prepared DegaFiles](https://github.com/broadinstitute/Celldega_Atera_v1_WTA_Kidney_non_diseased_FFPE)
as the [Atera kidney tutorial](https://broadinstitute.github.io/celldega/examples/tutorial_notebooks/Atera_Kidney_Landscape-Clustergram/).
The analysis downloads cell IDs and positions, two sparse gene vectors, and cell
areas (about 30 MB in total); the viewers stream images and spatial tiles as needed.

This example requires Celldega 0.28.0a6 or later:
`pip install --pre "celldega>=0.28.0a6"`.

In [1]:
import os
from pathlib import Path
from urllib.request import urlretrieve
import json

# Load the front end that matches this release.
os.environ["CELLDEGA_ESM_VERSION"] = "0.28.0-alpha.6"

import anndata as ad
import fsspec
import h5py
import numpy as np
import pandas as pd
from scipy import sparse
import celldega as dega

In [2]:
SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"
ANNDATA_URL = f"https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/{SAMPLE}.h5ad"

cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file()), None)
cache = repo / "data/atera_yearbook_review" if repo else cwd / "data/atera_yearbook_review"
cache.mkdir(parents=True, exist_ok=True)

for relative in ["landscape_parameters.json", "micron_to_image_transform.csv"]:
    path = cache / relative
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{relative}", path)
parameters = json.loads((cache / "landscape_parameters.json").read_text())
transform = np.loadtxt(cache / "micron_to_image_transform.csv")

## 1. Inspect the tissue context

This Landscape shows the whole section with its original default
clusters. Use its gene controls to examine EPCAM and PTPRC in tissue context;
the Yearbook below will focus on a small set of cells.

In [3]:
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera kidney — tissue context", width=0, height=600,
    max_tiles_to_view=25,
)
landscape.layout.width = "100%"
landscape

## 2. Select large cells that express both markers with `dega.select`

Build a small AnnData containing just EPCAM and PTPRC counts and the cell area for
all **399,120** cells. This pinned DegaFiles export uses numeric row positions in its
sparse gene files; the observation index below preserves the original cell IDs from
`cell_metadata.parquet`. Cell areas come from the `obs` table of the [annotated counts
H5AD](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/blob/main/Atera_v1_WTA_Kidney_non_diseased_FFPE.h5ad);
only the cell IDs and areas are read over HTTP, not the whole 1.31 GiB file.

`Selector.gene(...)` and `Selector.attr(...)` build predicates, `&` combines them, and
`selector.samplers.rank(...)` orders the matching cells. These thresholds are
exploratory settings that you can change. See the
[Select API](https://broadinstitute.github.io/celldega/python/select/api/) for more
query and sampling options.

In [4]:
MARKERS = ["EPCAM", "PTPRC"]
for relative in ["cell_metadata.parquet", *[f"cbg/{gene}.parquet" for gene in MARKERS]]:
    path = cache / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{relative}", path)

cell_ids = pd.Index(
    pd.read_parquet(cache / "cell_metadata.parquet", columns=["name"])["name"].astype(str),
    name="cell_id",
)
assert cell_ids.is_unique
assert parameters["use_int_index"], "This example expects the pinned indexed DegaFiles."

columns = []
for gene in MARKERS:
    frame = pd.read_parquet(cache / "cbg" / f"{gene}.parquet")
    rows = frame.index.to_numpy(dtype=np.int64)
    assert ((rows >= 0) & (rows < len(cell_ids))).all()
    columns.append(sparse.csc_matrix(
        (frame[gene].to_numpy(), (rows, np.zeros(len(rows), dtype=np.int64))),
        shape=(len(cell_ids), 1),
    ))

cells = ad.AnnData(
    X=sparse.hstack(columns, format="csr"),
    obs=pd.DataFrame(index=cell_ids),
    var=pd.DataFrame(index=MARKERS),
)
area_path = cache / "cell_area.parquet"
if not area_path.exists():
    # Read only the cell IDs and areas from the remote H5AD (about 20 MB of 1.31 GiB).
    with (
        fsspec.open(ANNDATA_URL, block_size=2**20, cache_type="blockcache") as remote,
        h5py.File(remote, "r") as h5,
    ):
        pd.DataFrame(
            {"cell_area": h5["obs/cell_area"][:]},
            index=pd.Index(h5["obs/_index"].asstr()[:], name="cell_id"),
        ).to_parquet(area_path)
cells.obs["cell_area"] = pd.read_parquet(area_path)["cell_area"].reindex(cells.obs_names)
assert cells.obs["cell_area"].notna().all()

print(f"{cells.n_obs:,} cells; {cells.n_vars} marker genes and cell area; original cell IDs retained")

399,120 cells; 2 marker genes and cell area; original cell IDs retained


In [5]:
selector = dega.select.Selector(cells)
mixed_markers = (selector.gene("EPCAM") >= 2) & (selector.gene("PTPRC") >= 2)
area_cutoff = cells.obs["cell_area"].quantile(0.9)  # the largest 10% of cells
large_cells = selector.attr("cell_area") >= area_cutoff
mixed_review = selector.select(
    query=mixed_markers & large_cells,
    sampler=selector.samplers.rank(attr=selector.attr("cell_area"), n=100, by="high"),
)
# To sample the matching candidates instead, use:
# sampler=selector.samplers.random(n=100, seed=0)

print(f"cell area >= {area_cutoff:.0f} µm²: {mixed_review.candidate_count:,} candidates; "
      f"{len(mixed_review)} cells for review")
mixed_review.to_dataframe().head()

cell area >= 111 µm²: 31 candidates; 31 cells for review


,id,rank,score
0,gcbdojlh-1,0,233.485855
1,meochnce-1,1,214.747375
2,ghniilni-1,2,213.437943
3,ghojilnh-1,3,187.836227
4,mkphobdg-1,4,179.663544


The counts support using these two genes as a screen. About 25% of cells have at
least 2 EPCAM counts and about 2% have at least 2 PTPRC counts. If the two genes
were unrelated, about 2,000 cells would pass both thresholds by chance; only 198
do, about a tenth as many, as expected for markers of separate lineages. Those 198
cells mostly have low counts (median 3 EPCAM and 2 PTPRC), which fits a few
transcripts from a neighboring cell better than a cell that truly expresses both
genes.

Two cells segmented as one should be larger than either cell alone, so the query
also keeps only the largest 10% of cells (at least 111 µm², above the 81 µm² median
of EPCAM-positive cells). That narrows the 198 cells to the 31 most likely to be
segmentation merges, ranked largest first.

EPCAM is typically strongest in distal tubule and collecting duct epithelium and
weak in proximal tubule, so this screen under-samples immune cells that border
proximal tubule, the largest epithelial population.

## 3. Review portraits across pages

Pass the candidate cell IDs to Yearbook. Click the checkbox at the lower right of a
portrait to select that cell. Inspect cell boundaries, nuclei, and marker transcripts
to decide which cells warrant follow-up: a single boundary enclosing two nuclei
suggests a segmentation merge; EPCAM transcripts concentrated along the edge an
immune cell shares with an epithelial neighbor suggest spillover.

In [6]:
review_selection = mixed_review
candidate_ids = review_selection.names()
yearbook = dega.viz.Yearbook(
    name="kidney_candidate_review", base_url=BASE_URL, cells=candidate_ids,
    rows=3, cols=5, portrait_size_um=50,
    dataset_name="Atera kidney — candidate review", width=0, height=850,
)
yearbook.layout.width = "100%"
yearbook

## 4. Get the selected cell IDs

`yearbook.selected_cells` returns the IDs of the cells you selected. Run this cell in
a live notebook after selecting portraits.

In [7]:
yearbook.selected_cells

[]

## Optional: sample high-count cells using observation metadata

A second `dega.select` example combines an `obs` threshold with seeded random
sampling. The [annotated counts H5AD](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/blob/main/Atera_v1_WTA_Kidney_non_diseased_FFPE.h5ad)
contains `total_counts` in `obs`. If you already downloaded it for the full kidney
tutorial, the code below reads only metadata in backed mode, not the count matrix.
It deliberately does **not** download that 1.31 GiB file just for this example.

Cells in the highest 1% of total counts are another review queue, not a doublet
classification. To inspect this alternative, replace `review_selection = mixed_review`
with `review_selection = high_count_review` in the Yearbook cell and rerun it.

In [8]:
source_path = (
    repo / "notebooks/data/atera_data" / SAMPLE / f"{SAMPLE}.h5ad"
    if repo else cwd / "data" / f"{SAMPLE}.h5ad"
)
high_count_review = None
if source_path.exists():
    source = ad.read_h5ad(source_path, backed="r")
    qc_cells = ad.AnnData(obs=source.obs[["total_counts"]].copy())
    source.file.close()
    qc_selector = dega.select.Selector(qc_cells)
    count_cutoff = qc_cells.obs["total_counts"].quantile(0.99)
    high_count_review = qc_selector.select(
        query=qc_selector.attr("total_counts") >= count_cutoff,
        sampler=qc_selector.samplers.random(n=100, seed=0),
    )
    print(f"total_counts >= {count_cutoff:g}: {high_count_review.candidate_count:,} candidates; "
          f"{len(high_count_review)} sampled for review")
else:
    print("Optional high-count example skipped: no cached H5AD. The marker-based review above is ready.")

total_counts >= 9225: 3,993 candidates; 100 sampled for review
